# Publication Figures — v2026.03.10

Interactive figures using `dashboard_patient_timeline_mv` and `risk_enriched_mv`.
Uses Plotly for interactive HTML and Altair for static publication-quality output.

**Data sources:**
- `risk_enriched_mv` — combined risk features with ETE, BRAF, staging
- `dashboard_patient_timeline_mv` — per-patient timeline events
- `survival_cohort_ready_mv` — time-to-event cohort
- `advanced_features_v3` — 60+ engineered features

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)

DB_PATH = Path("../thyroid_master_local.duckdb")
EXPORT_DIR = Path("../exports/THYROID_2026_PUBLICATION_BUNDLE_20260310_0414")

if DB_PATH.exists():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    print(f"Connected to local DuckDB: {DB_PATH}")
else:
    risk = pd.read_parquet(EXPORT_DIR / "risk_enriched_mv.parquet")
    timeline = pd.read_parquet(EXPORT_DIR / "dashboard_patient_timeline_mv.parquet")
    survival = pd.read_parquet(EXPORT_DIR / "survival_cohort_ready_mv.parquet")
    afv3 = pd.read_parquet(EXPORT_DIR / "advanced_features_v3.parquet")
    con = None
    print(f"Loaded parquet files from {EXPORT_DIR}")

def q(sql):
    return con.execute(sql).fetchdf() if con else None

## Figure 1 — AJCC 8th Stage Distribution

In [ ]:
if con:
    df_stage = q("SELECT ajcc_8th_stage, COUNT(*) AS n FROM risk_enriched_mv WHERE ajcc_8th_stage IS NOT NULL GROUP BY 1 ORDER BY 1")
else:
    df_stage = risk.groupby("ajcc_8th_stage").size().reset_index(name="n")

fig1 = px.bar(df_stage, x="ajcc_8th_stage", y="n",
              title="AJCC 8th Edition Stage Distribution",
              labels={"ajcc_8th_stage": "Stage", "n": "Patients"},
              color="ajcc_8th_stage")
fig1.update_layout(showlegend=False)
fig1.show()

## Figure 2 — Recurrence Risk by ETE Status

In [ ]:
if con:
    df_ete = q("""
        SELECT ete_status, recurrence_risk_category, COUNT(*) AS n
        FROM risk_enriched_mv
        WHERE ete_status IS NOT NULL AND recurrence_risk_category IS NOT NULL
        GROUP BY 1, 2 ORDER BY 1, 2
    """)
else:
    df_ete = risk.dropna(subset=["ete_status", "recurrence_risk_category"]).groupby(["ete_status", "recurrence_risk_category"]).size().reset_index(name="n")

fig2 = px.bar(df_ete, x="ete_status", y="n", color="recurrence_risk_category",
              barmode="group",
              title="Recurrence Risk Stratification by ETE Status",
              labels={"ete_status": "ETE", "n": "Patients", "recurrence_risk_category": "Risk"})
fig2.show()

## Figure 3 — Patient Timeline Event Density

In [ ]:
if con:
    df_tl = q("""
        SELECT event_type, COUNT(*) AS n
        FROM dashboard_patient_timeline_mv
        GROUP BY 1 ORDER BY 2 DESC LIMIT 15
    """)
else:
    df_tl = timeline.groupby("event_type").size().reset_index(name="n").nlargest(15, "n")

fig3 = px.bar(df_tl, x="n", y="event_type", orientation="h",
              title="Top 15 Clinical Event Types",
              labels={"event_type": "Event Type", "n": "Count"})
fig3.update_layout(yaxis={"categoryorder": "total ascending"})
fig3.show()

## Figure 4 — Kaplan-Meier Recurrence-Free Survival

Requires `lifelines`.

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

if con:
    df_surv = q("SELECT follow_up_days, recurrence_event, ajcc_8th_stage FROM survival_cohort_ready_mv WHERE follow_up_days IS NOT NULL")
else:
    df_surv = survival.dropna(subset=["follow_up_days"])

fig, ax = plt.subplots(figsize=(10, 6))
kmf = KaplanMeierFitter()
for stage in sorted(df_surv["ajcc_8th_stage"].dropna().unique()):
    mask = df_surv["ajcc_8th_stage"] == stage
    kmf.fit(df_surv.loc[mask, "follow_up_days"],
            event_observed=df_surv.loc[mask, "recurrence_event"],
            label=f"Stage {stage}")
    kmf.plot_survival_function(ax=ax)

ax.set_xlabel("Days from Surgery")
ax.set_ylabel("Recurrence-Free Survival")
ax.set_title("Kaplan-Meier: Recurrence-Free Survival by AJCC Stage")
plt.tight_layout()
plt.show()